# 한국 배당성장주 백테스트 (Google Colab 독립 실행)

이 노트북은 외부 API 키(DART API 키 등)나 별도 설정 없이 Google Colab에서 **`런타임 → 모두 실행`** 한 번으로 다음을 자동으로 수행합니다.

1. **한글 폰트 자동 설치·설정** (`NanumGothic`)
2. **공개 시점고정(Point-in-Time) 재무·배당 데이터 로드** (한국 상장사 374개사 `FY2018~2024` 재무·배당·자사주 + `FY2017~2025` 보정 실제 DPS + 국고채 10년물 금리)
3. **한국 증시 일별 수정주가·거래대금·KOSPI 실시간 수집** (네이버 금융 `fchart` 무인증 우선 → `FinanceDataReader` → `pykrx` 폴백)
4. **2가지 전략 동시 백테스트 및 시각화**
   - **전략 A (현행 정량 규칙 재현):** `rules.md` v1.23 핵심 1~9번 정량 규칙(과거 시점 자료 창 보정: 5년 배당 이력) + `밸류 할인 순위 + DPS 5년 성장률 순위` 상위 N종목 동일가중
   - **전략 B (배당 함정 보완안):** 전략 A + ① 배당수익률 2~12% 구간 ② DPS 5년 CAGR ≥ 3% ③ 60일 평균 일거래대금 ≥ 10억 원 ④ 52주 고점 대비 -35% 이하 & 최근 EPS 역성장 동시 차단 ⑤ 밸류 괴리율 ±50% 캡(윈저라이징)
5. **성과표 및 차트 4종 출력:** 누적 수익률(vs KOSPI), 일별 MDD, 연도별 세후 배당금 추이, 연도별 수익률 비교, 선정 종목 리스트

> **데이터 한계 고지:** 374개사 표본은 현재 상장 생존 기업 기준이므로 **생존 편향(Survivorship Bias)**이 존재할 수 있으며, KOSPI 일별 벤치마크는 배당 미포함 가격지수(PR)입니다.

In [ ]:
# ==============================================================================
# 1. 환경 설정, 패키지 설치 및 한글 폰트 자동 구성
# ==============================================================================
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 FinanceDataReader 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "finance-datareader"], check=False)

import re, gzip, json, math, time, statistics
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

# 한글 폰트 자동 탐색 및 즉시 등록 (Colab 런타임 재시작 없이 바로 적용)
def setup_korean_font():
    nanum_paths = [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
    ]
    chosen = None
    for p in nanum_paths:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()

    if chosen is None:
        available = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "Noto Sans CJK KR", "DejaVu Sans"]:
            if cand in available:
                chosen = cand
                break

    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"[완료] 폰트 설정: {FONT_NAME} | pandas {pd.__version__} | numpy {np.__version__}")
try:
    from IPython.display import display
except ImportError:
    display = print

In [ ]:
# ==============================================================================
# 2. 백테스트 파라미터 설정 (필요에 따라 자유롭게 수정 후 모두 실행)
# ==============================================================================
CONFIG = {
    # 백테스트 기준 사업연도 범위 (FY2019 -> 2020년 4월 첫 영업일 리밸런싱 시작)
    "START_FY": 2019,
    "END_FY": 2024,            # FY2024 -> 2025년 4월 리밸런싱 후 최신 거래일까지 보유

    # 포트폴리오 운용 가정
    "TOP_N": 10,               # 리밸런싱 시 상위 N종목 동일가중 편입
    "INITIAL_CAPITAL": 100_000_000,  # 초기 투자금 1억 원
    "FEE_RATE": 0.0025,        # 매매 편도 비용 0.25% (수수료 + 거래세 근사)
    "DIV_TAX_RATE": 0.154,     # 배당소득세 15.4%
    "REINVEST_DIVIDENDS": True,# 세후 배당금 재투자(DRIP) 여부

    # 품질 판정 결측 허용 (초기 연도 일부 항목 결측 시 최대 1개까지 허용; 엄격 모드는 0)
    "ALLOW_NA_COUNT": 1,

    # 전략 B (배당 함정 보완안) 추가 임계값
    "B_MIN_YIELD": 0.02,                # 최소 배당수익률 2.0%
    "B_MAX_YIELD": 0.12,                # 최대 배당수익률 12.0% (폭락형 초고배당 함정 차단)
    "B_MIN_DPS_CAGR_5Y": 0.03,          # 정기 DPS 5년 연평균 성장률 >= 3.0%
    "B_MIN_TURNOVER_60D": 1_000_000_000,# 60일 일평균 거래대금 >= 10억 원
    "B_MAX_DD_52W_WITH_EPS_DROP": -0.35,# 52주 고점 대비 -35% 이하 & 최근 EPS 역성장 동시 시 제외
    "B_VAL_CAP": 0.50,                  # 밸류에이션 개별 지표 괴리율 ±50% 캡(윈저라이징)

    # 공개 데이터 저장소 (로컬 data/ 우선, 없으면 GitHub Public Raw URL에서 자동 다운로드)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
    "CACHE_DIR": "./_cache_dividend_backtest",
}

print("[설정] 백테스트 기간: FY{START_FY}~FY{END_FY} (리밸런싱 {r0}-04 ~ {r1}-04 -> 현재)".format(
    START_FY=CONFIG["START_FY"], END_FY=CONFIG["END_FY"],
    r0=CONFIG["START_FY"] + 1, r1=CONFIG["END_FY"] + 1
))
print(f"[설정] 상위 {CONFIG['TOP_N']}종목 동일가중 | 초기자금 {CONFIG['INITIAL_CAPITAL']:,.0f}원 | "
      f"편도비용 {CONFIG['FEE_RATE']:.2%} | 배당세 {CONFIG['DIV_TAX_RATE']:.1%} | DRIP={CONFIG['REINVEST_DIVIDENDS']}")

In [ ]:
# ==============================================================================
# 3. 공개 Point-in-Time 재무·배당 데이터 및 국고채 10년물 로드
# ==============================================================================
CACHE_DIR = Path(CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def ensure_data_file(filename: str) -> Path:
    # 1) 로컬 실행 시 상대 경로 탐색
    for local_cand in [Path("../data") / filename, Path("data") / filename, Path("dividend-backtest/data") / filename]:
        if local_cand.exists():
            return local_cand

    # 2) 캐시 확인
    cached = CACHE_DIR / filename
    if cached.exists() and cached.stat().st_size > 0:
        return cached

    # 3) GitHub Public Raw URL 자동 다운로드 (브랜치 순차 탐색)
    last_err = None
    for br in CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{CONFIG['GITHUB_REPO']}/{br}/{CONFIG['DATA_SUBPATH']}/{filename}"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 0:
                cached.write_bytes(r.content)
                print(f"  [다운로드 완료] {filename} ({len(r.content):,} bytes, branch={br})")
                return cached
            last_err = f"HTTP {r.status_code} ({url})"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"데이터 파일({filename})을 가져오지 못했습니다: {last_err}")

pit_path = ensure_data_file("kr_fundamentals_pit.jsonl.gz")
rf_path = ensure_data_file("kr_treasury_10y_annual.csv")
kospi_snap_path = ensure_data_file("kospi_snapshots.json")

with gzip.open(pit_path, "rt", encoding="utf-8") as f:
    PIT_RECORDS = [json.loads(line) for line in f if line.strip()]

rf_df = pd.read_csv(rf_path)
RF_MAP = {int(row["연도"]): float(row["국고채10년_연평균_퍼센트"]) / 100.0 for _, row in rf_df.iterrows()}

with open(kospi_snap_path, encoding="utf-8") as f:
    KOSPI_SNAP = json.load(f)

mkt_counts = pd.Series([r["market"] for r in PIT_RECORDS]).value_counts().to_dict()
path_counts = pd.Series([r["path"] for r in PIT_RECORDS]).value_counts().to_dict()
print(f"[로드 완료] 상장사 {len(PIT_RECORDS)}개 | 시장={mkt_counts} | 업종경로={path_counts}")
print(f"[로드 완료] 국고채 10년물 연평균 금리: {min(RF_MAP)}~{max(RF_MAP)}년 (평균 {np.mean(list(RF_MAP.values())):.2%})")

In [ ]:
# ==============================================================================
# 4. 배당성장주 정량 판정 엔진 (1~9번 판정 + 전략 A/B 1차 품질 풀 산출)
# ==============================================================================
ERP = 0.05   # 주식위험프리미엄 5.0% 고정
TAX = 0.24   # 법인세율 24% 고정
REG_CAP = {
    "bank": ("cet1", 8.0, 2.0),
    "insurance": ("kics", 100.0, 50.0),
    "financial_holding": ("fh_capital", 100.0, 30.0),
    "securities": ("ncr", 100.0, 50.0),
}

def _f(v):
    if v is None:
        return None
    try:
        x = float(v)
        return None if (math.isnan(x) or math.isinf(x)) else x
    except (TypeError, ValueError):
        return None

def get_val(frame: dict, col: str, y: int):
    d = frame.get(col)
    return _f(d.get(str(y))) if isinstance(d, dict) else None

def avg3(frame: dict, col: str, end: int):
    vals = [v for y in (end - 2, end - 1, end) if (v := get_val(frame, col, y)) is not None]
    return sum(vals) / len(vals) if len(vals) >= 2 else None

def cagr_3avg(frame: dict, col: str, end: int, span: int):
    a, b = avg3(frame, col, end - span), avg3(frame, col, end)
    if a is None or a <= 0 or b is None:
        return None
    return -1.0 if b <= 0 else (b / a) ** (1.0 / span) - 1.0

def total_return_3y(frame: dict, end: int):
    W = (end - 2, end - 1, end)
    decl, buy, canc, acq, e3 = [], [], [], [], []
    for y in W:
        dps = get_val(frame, "dps_common", y)
        d = get_val(frame, "div_total_reported", y)
        if d is None and dps == 0:
            d = 0.0
        decl.append(d)
        b = get_val(frame, "treasury_purchase", y)
        has_fs = bool((frame.get("_fs_src") or {}).get(str(y)))
        buy.append(abs(b) if b is not None else (0.0 if has_fs else None))
        canc.append(get_val(frame, "ts_cancelled_common", y))
        acq.append(get_val(frame, "ts_acquired_common", y))
        nip = get_val(frame, "net_income_parent", y)
        nr = get_val(frame, "ni_reported", y)
        ni = get_val(frame, "net_income", y)
        e3.append(nip if nip is not None else (nr if nr is not None else ni))
    if any(v is None for v in decl) or any(v is None for v in e3):
        return None
    e_sum, d_sum = sum(e3), sum(decl)
    if any(v is None for v in buy):
        if any(v is None for v in canc) or sum(canc) > 0:
            return None
        num = d_sum
    else:
        b_sum = sum(buy)
        if b_sum == 0:
            s = 0.0
        else:
            k = sum(v for v in canc if v is not None)
            a = sum(v for v in acq if v is not None)
            s = min(1.0, k / a) if a > 0 else (1.0 if k > 0 else 0.0)
        num = d_sum + b_sum * s
    if e_sum <= 0:
        return {"tr": -1.0, "payout3": None, "num": num, "e3": e_sum}
    return {"tr": num / e_sum * 100.0, "payout3": d_sum / e_sum * 100.0, "num": num, "e3": e_sum}

def evaluate_company_pit(rec: dict, Y: int) -> dict | None:
    '''기준 사업연도 Y의 시점고정(Point-in-Time) 스냅샷으로 1~9번 판정 (미래 정보 누출 차단).'''
    frame = (rec.get("snapshots") or {}).get(str(Y))
    if not frame:
        return None
    price_info = rec.get("price") or {}
    ye_px = {int(k): _f(v) for k, v in (price_info.get("year_end") or {}).items()}
    t_px = _f((price_info.get("t_close") or {}).get(str(Y)))
    if not t_px or t_px <= 0:
        return None

    path = rec.get("path", "general")
    cap = rec.get("capital") or {}
    dps_y = get_val(frame, "dps_common", Y)
    if dps_y is None or dps_y <= 0:
        return None

    # 1번 배당 이력 (과거 시점 백테스트 자료 창 보정: 최근 5년 연속 지급·무삭감·DPS 5년 성장 > 0)
    paid_5y = all((get_val(frame, "dps_common", y) or 0) > 0 for y in range(Y - 4, Y + 1))
    cuts_5y = 0
    for y in range(Y - 4, Y + 1):
        a, b = get_val(frame, "dps_common", y - 1), get_val(frame, "dps_common", y)
        if a is not None and b is not None and b < a - min(0.5, abs(a) * 1e-6):
            cuts_5y += 1
    dps_g5 = cagr_3avg(frame, "dps_common", Y, 5)
    if not paid_5y or cuts_5y >= 2:
        v1 = "FAIL"
    elif cuts_5y == 0 and dps_g5 is not None and dps_g5 > 0:
        v1 = "PASS"
    else:
        v1 = "WARN"

    # 2번 EPS 성장 (3년 배당성향 30% 미만 시 DPS 초과 성장 면제)
    eps_g5 = cagr_3avg(frame, "eps_reported", Y, 5)
    tr = total_return_3y(frame, Y)
    p3 = tr["payout3"] if tr else None
    if eps_g5 is None or dps_g5 is None:
        v2 = "NA"
    elif eps_g5 < 0 and dps_g5 > 0:
        v2 = "FAIL"
    elif eps_g5 > 0 and (dps_g5 <= eps_g5 + 0.03 or (p3 is not None and p3 < 30.0)):
        v2 = "PASS"
    else:
        v2 = "WARN"

    # 3번 FCF(일반) / 순이익(금융) 꾸준함
    col3 = "fcf" if path == "general" else "net_income_parent"
    vals3 = [get_val(frame, col3, y) for y in range(Y - 9, Y + 1)]
    if path != "general":
        vals3 = [v if v is not None else get_val(frame, "ni_reported", y) for v, y in zip(vals3, range(Y - 9, Y + 1))]
    known3 = [v for v in vals3 if v is not None]
    neg3, pos3 = sum(1 for v in known3 if v < 0), sum(1 for v in known3 if v > 0)
    if neg3 >= 3:
        v3 = "FAIL"
    elif len(known3) >= 4 and pos3 / len(known3) >= 0.8:
        v3 = "PASS"
    elif len(known3) >= 4:
        v3 = "WARN"
    else:
        v3 = "NA"

    # 4번 자본효율 (일반 ROIC-WACC >= 3%p, 금융 ROE-COE >= 1%p)
    spreads = []
    for y in range(Y - 4, Y + 1):
        rf = RF_MAP.get(y, 0.03)
        coe = rf + ERP
        if path == "general":
            r_val = get_val(frame, "roic", y)
            if r_val is None:
                continue
            px_y = ye_px.get(y)
            sh_y = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
            debt_y = get_val(frame, "debt", y) or 0.0
            mcap_y = px_y * sh_y if (px_y and sh_y) else None
            if mcap_y and mcap_y > 0:
                tot = mcap_y + debt_y
                int_y = get_val(frame, "interest_expense", y) or get_val(frame, "interest_paid", y)
                kd = (min(0.15, abs(int_y) / debt_y) if (debt_y > 0 and int_y and int_y > 0) else 0.04) * (1 - TAX)
                wacc = coe * (mcap_y / tot) + kd * (debt_y / tot)
            else:
                wacc = coe
            spreads.append((y, r_val - wacc))
        else:
            r_val = get_val(frame, "roe", y)
            if r_val is not None:
                spreads.append((y, r_val - coe))
    pass_thr, fail_thr = (0.03, 0.0) if path == "general" else (0.01, -0.01)
    avg5_sp = sum(sp for _, sp in spreads) / len(spreads) if len(spreads) >= 3 else None
    last3_sp = [sp for y, sp in spreads if y >= Y - 2]
    if len(last3_sp) == 3 and all(sp < fail_thr for sp in last3_sp):
        v4 = "FAIL"
    elif avg5_sp is not None:
        rec2_sp = [sp for y, sp in spreads if y >= Y - 1]
        v4 = "PASS" if (avg5_sp >= pass_thr and (not rec2_sp or sum(rec2_sp) / len(rec2_sp) >= avg5_sp - 0.03)) else "WARN"
    else:
        v4 = "NA"

    # 5번 주주환원 수준 (3년 총환원율 >= 30% & 배당성향 <= 60%, 금융 상한 추가)
    if tr is None:
        v5 = "NA"
    elif tr["e3"] <= 0 or tr["tr"] < 20.0 or (p3 is not None and p3 > 75.0) or (path != "general" and tr["tr"] > 100.0):
        v5 = "FAIL"
    elif tr["tr"] >= 30.0 and (p3 is not None and p3 <= 60.0) and (path == "general" or tr["tr"] <= 66.7):
        v5 = "PASS"
    else:
        v5 = "WARN"

    # 6번 자사주·희석 (FY2019는 DART 주식총수 개시연도 제약으로 Y-4 보완 허용)
    sh_now = get_val(frame, "outstanding_common", Y)
    sh_base = get_val(frame, "outstanding_common", Y - 5) or (get_val(frame, "outstanding_common", Y - 4) if Y == 2019 else None)
    if sh_now is None or not sh_base:
        v6 = "NA"
    else:
        chg = sh_now / sh_base - 1.0
        canc5 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(Y - 4, Y + 1))
        acq5 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(Y - 4, Y + 1))
        if chg >= 0.03:
            v6 = "FAIL"
        elif (chg < -0.01 and canc5 > 0) or (-0.01 <= chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)):
            v6 = "PASS"
        else:
            v6 = "WARN"

    # 7번 환원 감당 (일반 경로만 품질 종합에 산입)
    if path != "general":
        v7 = "PASS"
    else:
        fcfs = [get_val(frame, "fcf", y) for y in (Y - 2, Y - 1, Y)]
        if tr is None or any(v is None for v in fcfs):
            v7 = "NA"
        else:
            fcf3 = sum(fcfs)
            if fcf3 <= 0:
                v7 = "FAIL" if tr["num"] > 0 else "PASS"
            else:
                cov = tr["num"] / fcf3
                v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2 / 3 else "PASS")

    # 8번 재무건전성 (일반: 순부채/EBITDA & ICR, 금융: 규제자본비율)
    if path == "general":
        nd, nde = get_val(frame, "net_debt", Y), get_val(frame, "net_debt_to_ebitda", Y)
        icr, eb = get_val(frame, "interest_coverage", Y), get_val(frame, "ebitda", Y)
        if nd is not None and nd <= 0:
            v8 = "WARN" if (eb is not None and eb <= 0) else "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0):
            v8 = "FAIL"
        elif nde is not None and icr is not None:
            v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else:
            v8 = "NA"
    else:
        mk, req, margin = REG_CAP.get(path, ("cet1", 8.0, 2.0))
        cval = _f((cap.get(str(Y)) or {}).get(mk))
        v8 = "NA" if cval is None else ("PASS" if cval >= req + margin else ("WARN" if cval >= req else "FAIL"))

    verdicts = [v1, v2, v3, v4, v5, v6, v8] if path != "general" else [v1, v2, v3, v4, v5, v6, v7, v8]
    n_fail, n_warn, n_na = verdicts.count("FAIL"), verdicts.count("WARN"), verdicts.count("NA")
    qual_pass = (n_fail == 0 and n_warn <= 2 and n_na <= CONFIG["ALLOW_NA_COUNT"])

    # 9번 밸류에이션 (후행 FY 기준 vs 과거 연말 중앙값)
    eps_y = get_val(frame, "eps_reported", Y)
    eps_prev = get_val(frame, "eps_reported", Y - 1)
    eq_y = get_val(frame, "equity_parent", Y) or get_val(frame, "equity", Y)
    sh_tot = (get_val(frame, "outstanding_common", Y) or 0) + (get_val(frame, "outstanding_pref", Y) or 0)
    bps_y = eq_y / sh_tot if (eq_y and sh_tot > 0) else None

    cur_per = t_px / eps_y if (eps_y and eps_y > 0) else None
    cur_pbr = t_px / bps_y if (bps_y and bps_y > 0) else None
    cur_yld = dps_y / t_px if (dps_y and dps_y > 0) else None

    hist_per, hist_pbr, hist_yld = [], [], []
    for y in range(Y - 9, Y + 1):
        px = ye_px.get(y)
        if not px or px <= 0:
            continue
        e, d = get_val(frame, "eps_reported", y), get_val(frame, "dps_common", y)
        eq = get_val(frame, "equity_parent", y) or get_val(frame, "equity", y)
        sh = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
        b = eq / sh if (eq and sh > 0) else None
        if e and e > 0: hist_per.append(px / e)
        if b and b > 0: hist_pbr.append(px / b)
        if d and d > 0: hist_yld.append(d / px)

    cap_lim = CONFIG["B_VAL_CAP"]
    devs_raw, devs_cap = [], []
    for cur_v, hist_v, higher_is_cheap in ((cur_per, hist_per, False), (cur_pbr, hist_pbr, False), (cur_yld, hist_yld, True)):
        if cur_v is not None and len(hist_v) >= 4:
            med = statistics.median(hist_v)
            if med > 0:
                dev = (cur_v / med - 1.0) if higher_is_cheap else -(cur_v / med - 1.0)
                devs_raw.append(dev)
                devs_cap.append(max(-cap_lim, min(cap_lim, dev)))

    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None

    return {
        "fy": Y, "rebal_year": Y + 1, "ticker": rec["ticker"], "name": rec["name"],
        "market": rec.get("market", "KOSPI"), "path": path,
        "qual_pass": qual_pass, "n_fail": n_fail, "n_warn": n_warn, "n_na": n_na,
        "val_raw": val_raw, "val_cap": val_cap,
        "dps_g5": dps_g5, "eps_g5": eps_g5, "cur_yld": cur_yld, "dps_y": dps_y,
        "eps_down": bool(eps_y is not None and eps_prev is not None and eps_y < eps_prev),
        "spread": avg5_sp if avg5_sp is not None else -9.0,
        "tr_3y": tr["tr"] if tr else None,
        "payout_3y": p3,
    }

# 전 연도(FY2019~FY2024) 1차 품질 통과 후보군 산출
YEARLY_POOL = {}
NEEDED_TICKERS = set()
for fy in range(CONFIG["START_FY"], CONFIG["END_FY"] + 1):
    rows = [x for r in PIT_RECORDS if (x := evaluate_company_pit(r, fy)) is not None]
    pool = [x for x in rows if x["qual_pass"] and x["val_raw"] is not None]
    YEARLY_POOL[fy] = pool
    NEEDED_TICKERS.update(x["ticker"] for x in pool)
    print(f"  FY{fy} (리밸런싱 {fy+1}-04): 품질 통과 후보 {len(pool):2d}종목")

print(f"\n[완료] 일별 시세 수집 대상 고유 종목 수: {len(NEEDED_TICKERS)}개 (+ KOSPI 지수)")

In [ ]:
# ==============================================================================
# 5. 한국 증시 일별 수정주가·거래대금·KOSPI 실시간 수집 및 전략 A/B 최종 종목 선정
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-backtest-colab)"}

def fetch_naver_ohlcv(symbol: str, count: int = 3200) -> pd.DataFrame:
    '''네이버 금융 fchart API (무인증, 수정주가 일봉). symbol: 6자리 티커 또는 "KOSPI".'''
    last_err = None
    for attempt in range(3):
        try:
            r = requests.get(
                _FCHART_URL,
                params={"symbol": symbol, "timeframe": "day", "count": count, "requestType": 0},
                headers=_HEADERS, timeout=20,
            )
            r.raise_for_status()
            rows = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                d, o, h, l, c, v = m.group(1).split("|")
                rows.append((pd.Timestamp(d), float(o), float(h), float(l), float(c), float(v)))
            df = pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume"]).set_index("date")
            df = df[df["close"] > 0].sort_index()
            if not df.empty:
                return df
            last_err = ValueError("empty naver response")
        except Exception as e:
            last_err = e
            time.sleep(0.5 * (attempt + 1))
    raise RuntimeError(f"Naver fchart 실패({symbol}): {last_err}")

def fetch_daily_ohlcv_resilient(symbol: str, start: str = "2018-01-01") -> pd.DataFrame:
    '''1순위: 네이버 fchart -> 2순위: FinanceDataReader -> 3순위: pykrx 폴백.'''
    try:
        df = fetch_naver_ohlcv(symbol)
        return df.loc[df.index >= pd.Timestamp(start)]
    except Exception:
        pass

    try:
        import FinanceDataReader as fdr
        fdr_sym = "KS11" if symbol == "KOSPI" else symbol
        df = fdr.DataReader(fdr_sym, start)
        col_map = {c: c.lower() for c in df.columns}
        df = df.rename(columns=col_map)
        for col in ["open", "high", "low", "close", "volume"]:
            if col not in df.columns:
                df[col] = df["close"] if col != "volume" else 0.0
        df = df[["open", "high", "low", "close", "volume"]]
        df = df[df["close"] > 0].sort_index()
        if not df.empty:
            return df
    except Exception:
        pass

    try:
        from pykrx import stock as krx_stock
        end_str = datetime.today().strftime("%Y%m%d")
        start_str = start.replace("-", "")
        if symbol == "KOSPI":
            df = krx_stock.get_index_ohlcv(start_str, end_str, "1001")
            df = df.rename(columns={"시가": "open", "고가": "high", "저가": "low", "종가": "close", "거래량": "volume"})
        else:
            df = krx_stock.get_market_ohlcv(start_str, end_str, symbol, adjusted=True)
            df = df.rename(columns={"시가": "open", "고가": "high", "저가": "low", "종가": "close", "거래량": "volume"})
        df = df[["open", "high", "low", "close", "volume"]]
        df = df[df["close"] > 0].sort_index()
        if not df.empty:
            return df
    except Exception:
        pass

    raise RuntimeError(f"모든 시세 수집 경로 실패: {symbol}")

print("[수집] KOSPI 및 후보 종목 일별 시세 수집 중...")
t0 = time.time()
KOSPI_DF = fetch_daily_ohlcv_resilient("KOSPI", start="2018-01-01")

PRICE_DATA = {}
failed_tickers = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = {ex.submit(fetch_daily_ohlcv_resilient, t, "2018-01-01"): t for t in sorted(NEEDED_TICKERS)}
    for fut in as_completed(futs):
        t = futs[fut]
        try:
            PRICE_DATA[t] = fut.result()
        except Exception as e:
            failed_tickers.append((t, str(e)))

print(f"[완료] {len(PRICE_DATA)}/{len(NEEDED_TICKERS)} 종목 + KOSPI 수집 완료 ({time.time() - t0:.1f}초)")
print(f"       KOSPI 일별 범위: {KOSPI_DF.index.min().date()} ~ {KOSPI_DF.index.max().date()} ({len(KOSPI_DF):,} 거래일)")
if failed_tickers:
    print(f"  [주의] 시세 수집 실패 {len(failed_tickers)}건: {failed_tickers[:5]}")

# 각 리밸런싱 시점(매년 4월 첫 영업일) 계산 및 전략 A / 전략 B 최종 종목 선정
TRADING_DAYS = KOSPI_DF.index

def first_trading_day_in_april(year: int) -> pd.Timestamp:
    cands = TRADING_DAYS[(TRADING_DAYS.year == year) & (TRADING_DAYS.month == 4)]
    if len(cands) == 0:
        raise ValueError(f"{year}년 4월 거래일이 없습니다.")
    return cands[0]

def compute_market_guards(ticker: str, as_of: pd.Timestamp) -> tuple[float | None, float | None, float | None]:
    '''as_of 당일(포함) 이전 일봉만 사용해 60일 일평균 거래대금, 52주 고점 대비 낙폭, 당일 종가 산출 (미래 누수 차단).'''
    df = PRICE_DATA.get(ticker)
    if df is None or df.empty:
        return None, None, None
    sub = df.loc[df.index <= as_of]
    if len(sub) < 20:
        return None, None, None
    cur_close = float(sub["close"].iloc[-1])
    turnover_60d = float((sub["close"] * sub["volume"]).tail(60).mean())
    w52 = sub.loc[sub.index >= as_of - pd.Timedelta(days=365)]
    high_52w = float(w52["high"].max()) if not w52.empty else cur_close
    dd_52w = (cur_close / high_52w - 1.0) if high_52w > 0 else 0.0
    return turnover_60d, dd_52w, cur_close

def rank_and_pick(cands: list[dict], val_key: str, top_n: int) -> list[dict]:
    if not cands:
        return []
    vals = [c[val_key] for c in cands]
    grs = [c["dps_g5"] if c["dps_g5"] is not None else -99.0 for c in cands]
    scored = []
    for c in cands:
        vr = 1 + sum(1 for v in vals if v > c[val_key])
        gr = 1 + sum(1 for g in grs if g > (c["dps_g5"] if c["dps_g5"] is not None else -99.0))
        c_copy = dict(c)
        c_copy["val_rank"] = vr
        c_copy["growth_rank"] = gr
        c_copy["rank_sum"] = vr + gr
        scored.append((vr + gr, -c[val_key], -c["spread"], c_copy))
    scored.sort(key=lambda x: x[:3])
    return [x[3] for x in scored[:top_n]]

PICKS_A, PICKS_B = {}, {}
FILTERED_OUT_B = []

for fy in range(CONFIG["START_FY"], CONFIG["END_FY"] + 1):
    rebal_dt = first_trading_day_in_april(fy + 1)
    enriched = []
    for c in YEARLY_POOL[fy]:
        t = c["ticker"]
        to60, dd52, px_rebal = compute_market_guards(t, rebal_dt)
        if px_rebal is None:
            continue
        item = dict(c)
        item["rebal_date"] = rebal_dt
        item["rebal_close"] = px_rebal
        item["turnover_60d"] = to60
        item["dd_52w"] = dd52
        item["rebal_yld"] = item["dps_y"] / px_rebal if px_rebal > 0 else None

        # 전략 B 배당 함정 필터 판정
        reasons = []
        if item["rebal_yld"] is None or item["rebal_yld"] < CONFIG["B_MIN_YIELD"]:
            reasons.append(f"저배당({(item['rebal_yld'] or 0):.1%}<{CONFIG['B_MIN_YIELD']:.0%})")
        elif item["rebal_yld"] > CONFIG["B_MAX_YIELD"]:
            reasons.append(f"초고배당함정({item['rebal_yld']:.1%}>{CONFIG['B_MAX_YIELD']:.0%})")
        if item["dps_g5"] is None or item["dps_g5"] < CONFIG["B_MIN_DPS_CAGR_5Y"]:
            reasons.append(f"저성장({(item['dps_g5'] or 0):.1%}<{CONFIG['B_MIN_DPS_CAGR_5Y']:.0%})")
        if to60 is None or to60 < CONFIG["B_MIN_TURNOVER_60D"]:
            reasons.append(f"저유동성({(to60 or 0)/1e8:.1f}억)")
        if dd52 is not None and dd52 <= CONFIG["B_MAX_DD_52W_WITH_EPS_DROP"] and item["eps_down"]:
            reasons.append(f"급락+이익역성장(52주 {dd52:.1%})")
        item["b_pass"] = (len(reasons) == 0)
        item["b_reject_reason"] = ", ".join(reasons)
        enriched.append(item)

    picks_a = rank_and_pick(enriched, "val_raw", CONFIG["TOP_N"])
    picks_b = rank_and_pick([x for x in enriched if x["b_pass"]], "val_cap", CONFIG["TOP_N"])
    PICKS_A[rebal_dt] = picks_a
    PICKS_B[rebal_dt] = picks_b

    b_tickers = {x["ticker"] for x in picks_b}
    for x in picks_a:
        if not x["b_pass"]:
            FILTERED_OUT_B.append({
                "리밸런싱일": rebal_dt.strftime("%Y-%m-%d"),
                "종목명": x["name"], "티커": x["ticker"],
                "배당수익률": f"{(x['rebal_yld'] or 0):.2%}",
                "DPS 5년성장": f"{(x['dps_g5'] or 0):.1%}",
                "52주 낙폭": f"{(x['dd_52w'] or 0):.1%}",
                "60일 거래대금(억)": round((x["turnover_60d"] or 0) / 1e8, 1),
                "전략 B 제외 사유": x["b_reject_reason"],
            })

    print(f"  {rebal_dt.date()} (FY{fy}): 전략 A {len(picks_a):2d}종목 / 전략 B {len(picks_b):2d}종목 (전체 통과 A={len(enriched)}, B={sum(x['b_pass'] for x in enriched)})")

In [ ]:
# ==============================================================================
# 6. 일별 포트폴리오 백테스트 엔진 (배당 재투자 DRIP + 거래비용 + 세후 배당금 추적)
# ==============================================================================
# 종목별 연도별 실제 보정 DPS 매핑 (FY2017~FY2025)
ACT_DPS_MAP = {r["ticker"]: {int(y): float(v) for y, v in (r.get("act_dps") or {}).items()} for r in PIT_RECORDS}

def get_dps_for_calendar_year(ticker: str, year: int) -> tuple[float, bool]:
    '''결산연도 year의 실제 보정 DPS 반환. 아직 공시 전인 최신 연도는 직전 연도 DPS로 보수적 추정(추정 여부 표기).'''
    dmap = ACT_DPS_MAP.get(ticker) or {}
    if year in dmap:
        return dmap[year], False
    prev_years = [y for y in dmap if y < year]
    if prev_years:
        return dmap[max(prev_years)], True
    return 0.0, True

def run_daily_backtest(picks_by_date: dict[pd.Timestamp, list[dict]], label: str) -> dict:
    rebal_dates = sorted(picks_by_date.keys())
    start_dt = rebal_dates[0]
    cal = TRADING_DAYS[TRADING_DAYS >= start_dt]

    # 가격 패널 (전방 채움으로 휴장/거래정지일 직전 종가 유지)
    all_tickers = sorted({x["ticker"] for picks in picks_by_date.values() for x in picks})
    close_panel = pd.DataFrame(index=cal)
    for t in all_tickers:
        if t in PRICE_DATA:
            close_panel[t] = PRICE_DATA[t]["close"].reindex(cal).ffill()

    # 연도별 마지막 거래일 (12월 결산 배당 권리 확정일 기준)
    year_last_day = {}
    for y, grp in pd.Series(cal, index=cal).groupby(cal.year):
        if grp.iloc[-1].month == 12:
            year_last_day[y] = grp.iloc[-1]

    cash = float(CONFIG["INITIAL_CAPITAL"])
    shares = {}  # ticker -> float shares
    rebal_set = set(rebal_dates)
    fee = CONFIG["FEE_RATE"]
    div_tax = CONFIG["DIV_TAX_RATE"]
    reinvest = CONFIG["REINVEST_DIVIDENDS"]

    nav_series = []
    cash_div_by_year = {}
    div_yield_by_year = {}
    est_flag_by_year = {}
    turnover_log = []

    for dt in cal:
        # 현재 평가액 계산
        stock_val = 0.0
        for t, sh in shares.items():
            px = close_panel.at[dt, t] if t in close_panel.columns else np.nan
            if not np.isnan(px) and px > 0:
                stock_val += sh * px
        total_val = cash + stock_val

        # 리밸런싱일 처리 (매년 4월 첫 영업일)
        if dt in rebal_set:
            picks = [x for x in picks_by_date[dt] if x["ticker"] in close_panel.columns and pd.notna(close_panel.at[dt, x["ticker"]])]
            if picks:
                investable = total_val if reinvest else (stock_val if shares else total_val)
                target_w = {x["ticker"]: 1.0 / len(picks) for x in picks}
                cur_w = {}
                if investable > 0:
                    for t, sh in shares.items():
                        px = close_panel.at[dt, t]
                        if pd.notna(px) and px > 0:
                            cur_w[t] = (sh * px) / investable
                # 포지션 변화분 기준 거래비용 (매도 + 매수 각각 편도 수수료 적용)
                all_t = set(target_w) | set(cur_w)
                one_way_turnover = 0.5 * sum(abs(target_w.get(t, 0.0) - cur_w.get(t, 0.0)) for t in all_t)
                trade_cost = investable * (sum(abs(target_w.get(t, 0.0) - cur_w.get(t, 0.0)) for t in all_t)) * fee
                turnover_log.append((dt, one_way_turnover, trade_cost))

                net_investable = max(0.0, investable - trade_cost)
                new_shares = {}
                for t, w in target_w.items():
                    px = close_panel.at[dt, t]
                    new_shares[t] = (net_investable * w) / px
                shares = new_shares
                if reinvest:
                    cash = 0.0
                else:
                    # 배당 미재투자 시 누적 현금은 그대로 유지, 주식 원금만 거래비용 차감
                    cash = max(0.0, total_val - investable)
                total_val = cash + sum(shares[t] * close_panel.at[dt, t] for t in shares)

        # 연말 결산 배당 권리 확정 (12월 마지막 거래일 보유 주식 기준 세후 배당금 산출 -> 익년 4월 리밸런싱 시 재투자)
        if dt == year_last_day.get(dt.year) and shares:
            gross_div = 0.0
            any_est = False
            for t, sh in shares.items():
                px = close_panel.at[dt, t]
                dps, is_est = get_dps_for_calendar_year(t, dt.year)
                # 분할/증자 이상치 방어: 단일 종목 배당수익률 25% 초과는 캡
                if pd.notna(px) and px > 0 and dps / px > 0.25:
                    dps = px * 0.25
                gross_div += sh * dps
                any_est = any_est or is_est
            net_div = gross_div * (1.0 - div_tax)
            cash_div_by_year[dt.year] = net_div
            div_yield_by_year[dt.year] = net_div / total_val if total_val > 0 else 0.0
            est_flag_by_year[dt.year] = any_est
            cash += net_div
            total_val += net_div

        nav_series.append((dt, total_val))

    nav = pd.Series({d: v for d, v in nav_series}, name=label)
    dd = nav / nav.cummax() - 1.0
    return {
        "label": label,
        "nav": nav,
        "dd": dd,
        "annual_div": pd.Series(cash_div_by_year, dtype=float),
        "annual_div_yield": pd.Series(div_yield_by_year, dtype=float),
        "est_flag": est_flag_by_year,
        "turnover": turnover_log,
    }

RES_A = run_daily_backtest(PICKS_A, "전략 A (현행 정량 규칙)")
RES_B = run_daily_backtest(PICKS_B, "전략 B (배당 함정 보완안)")

# KOSPI 벤치마크 (동일 거래일 기준 정규화)
kospi_cal = KOSPI_DF["close"].reindex(RES_A["nav"].index).ffill()
KOSPI_NAV = kospi_cal / kospi_cal.iloc[0] * CONFIG["INITIAL_CAPITAL"]
KOSPI_DD = KOSPI_NAV / KOSPI_NAV.cummax() - 1.0

print(f"[백테스트 완료] 기간: {RES_A['nav'].index[0].date()} ~ {RES_A['nav'].index[-1].date()} ({len(RES_A['nav']):,} 거래일)")
print(f"  전략 A 최종 평가액: {RES_A['nav'].iloc[-1]:,.0f}원 ({RES_A['nav'].iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")
print(f"  전략 B 최종 평가액: {RES_B['nav'].iloc[-1]:,.0f}원 ({RES_B['nav'].iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")
print(f"  KOSPI  최종 평가액: {KOSPI_NAV.iloc[-1]:,.0f}원 ({KOSPI_NAV.iloc[-1]/CONFIG['INITIAL_CAPITAL']-1:+.1%})")

In [ ]:
# ==============================================================================
# 7. 핵심 성과표 출력 (CAGR, 누적수익률, MDD, 연변동성, 샤프지수, 배당수익률)
# ==============================================================================
# 백테스트 기간 평균 무위험수익률 (한국은행 국고채 10년물 연평균 금리)
bt_years = sorted(set(RES_A["nav"].index.year))
RF_AVG = float(np.mean([RF_MAP.get(y, RF_MAP[max(RF_MAP)]) for y in bt_years]))

def calc_metrics(nav: pd.Series, dd: pd.Series, kospi_nav: pd.Series, annual_div_yield: pd.Series | None = None) -> dict:
    days = (nav.index[-1] - nav.index[0]).days
    years = days / 365.25
    tot_ret = nav.iloc[-1] / nav.iloc[0] - 1.0
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1.0 / years) - 1.0
    daily_ret = nav.pct_change().dropna()
    ann_vol = float(daily_ret.std() * np.sqrt(252))
    sharpe = (cagr - RF_AVG) / ann_vol if ann_vol > 0 else np.nan
    mdd = float(dd.min())
    mdd_dt = dd.idxmin().strftime("%Y-%m-%d")
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    # 연도별 수익률 및 코스피 대비 승률
    yr_ret = nav.groupby(nav.index.year).last().pct_change().dropna()
    k_yr_ret = kospi_nav.groupby(kospi_nav.index.year).last().pct_change().dropna()
    common_y = yr_ret.index.intersection(k_yr_ret.index)
    win_rate = float((yr_ret.loc[common_y] > k_yr_ret.loc[common_y]).mean()) if len(common_y) else np.nan

    avg_dy = float(annual_div_yield.mean()) if (annual_div_yield is not None and not annual_div_yield.empty) else np.nan
    return {
        "최종 평가액 (1억 기준)": f"{nav.iloc[-1]:,.0f}원",
        "누적 수익률": f"{tot_ret:+.1%}",
        "CAGR (연평균)": f"{cagr:+.2%}",
        "MDD (최대낙폭)": f"{mdd:.1%} ({mdd_dt})",
        "연환산 변동성": f"{ann_vol:.1%}",
        f"샤프지수 (Rf={RF_AVG:.2%})": f"{sharpe:.2f}",
        "칼마지수 (CAGR/|MDD|)": f"{calmar:.2f}",
        "연간 승률 (vs KOSPI)": "-" if np.isnan(win_rate) else f"{win_rate:.0%}",
        "평균 세후 배당수익률": "-" if np.isnan(avg_dy) else f"{avg_dy:.2%}",
    }

summary_df = pd.DataFrame({
    "전략 A (현행 규칙)": calc_metrics(RES_A["nav"], RES_A["dd"], KOSPI_NAV, RES_A["annual_div_yield"]),
    "전략 B (함정 보완안)": calc_metrics(RES_B["nav"], RES_B["dd"], KOSPI_NAV, RES_B["annual_div_yield"]),
    "KOSPI (가격지수 PR)": calc_metrics(KOSPI_NAV, KOSPI_DD, KOSPI_NAV, None),
})

print("■ 핵심 성과 비교표 (일별 수정주가 기준)")
display(summary_df)

In [ ]:
# ==============================================================================
# 8. 시각화 대시보드 (누적 수익률 · MDD · 연도별 배당금 추이 · 구간별 수익률)
# ==============================================================================
COLOR_A = "#1f77b4"   # 파랑 (전략 A)
COLOR_B = "#d62728"   # 빨강 (전략 B)
COLOR_K = "#6c757d"   # 회색 (KOSPI)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("한국 배당성장주 백테스트 성과 대시보드 (전략 A vs 전략 B vs KOSPI)", fontsize=15, fontweight="bold", y=0.98)

# (1) 일별 누적 수익률 추이
ax1 = axes[0, 0]
ret_a = (RES_A["nav"] / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ret_b = (RES_B["nav"] / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100
ret_k = (KOSPI_NAV / CONFIG["INITIAL_CAPITAL"] - 1.0) * 100

ax1.plot(ret_a.index, ret_a, label=f"전략 A 현행 ({ret_a.iloc[-1]:+.1f}%)", color=COLOR_A, lw=2.0)
ax1.plot(ret_b.index, ret_b, label=f"전략 B 보완 ({ret_b.iloc[-1]:+.1f}%)", color=COLOR_B, lw=2.2)
ax1.plot(ret_k.index, ret_k, label=f"KOSPI PR ({ret_k.iloc[-1]:+.1f}%)", color=COLOR_K, lw=1.6, ls="--")
ax1.axhline(0, color="black", lw=0.7, alpha=0.5)
ax1.set_title("① 일별 누적 수익률 (배당 재투자 · 비용/세금 반영)", fontsize=12, fontweight="bold")
ax1.set_ylabel("누적 수익률 (%)")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax1.legend(loc="upper left", frameon=True)

# (2) 일별 최대 낙폭 (Drawdown / MDD)
ax2 = axes[0, 1]
dd_a = RES_A["dd"] * 100
dd_b = RES_B["dd"] * 100
dd_k = KOSPI_DD * 100

ax2.fill_between(dd_a.index, dd_a, 0, color=COLOR_A, alpha=0.18)
ax2.plot(dd_a.index, dd_a, label=f"전략 A (MDD {dd_a.min():.1f}%)", color=COLOR_A, lw=1.6)
ax2.fill_between(dd_b.index, dd_b, 0, color=COLOR_B, alpha=0.18)
ax2.plot(dd_b.index, dd_b, label=f"전략 B (MDD {dd_b.min():.1f}%)", color=COLOR_B, lw=1.8)
ax2.plot(dd_k.index, dd_k, label=f"KOSPI (MDD {dd_k.min():.1f}%)", color=COLOR_K, lw=1.4, ls="--")
ax2.set_title("② 일별 최대 낙폭 (Drawdown / MDD)", fontsize=12, fontweight="bold")
ax2.set_ylabel("고점 대비 낙폭 (%)")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax2.legend(loc="lower left", frameon=True)

# (3) 연도별 세후 배당금 수령액 추이 (초기 투자금 1억 원 기준)
ax3 = axes[1, 0]
div_years = sorted(set(RES_A["annual_div"].index) | set(RES_B["annual_div"].index))
x = np.arange(len(div_years))
w = 0.36
div_a_man = [RES_A["annual_div"].get(y, 0.0) / 1e4 for y in div_years]  # 만원 단위
div_b_man = [RES_B["annual_div"].get(y, 0.0) / 1e4 for y in div_years]

bars_a = ax3.bar(x - w / 2, div_a_man, width=w, label="전략 A (만원)", color=COLOR_A, alpha=0.88)
bars_b = ax3.bar(x + w / 2, div_b_man, width=w, label="전략 B (만원)", color=COLOR_B, alpha=0.88)
for bars in (bars_a, bars_b):
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax3.text(bar.get_x() + bar.get_width() / 2, h + 12, f"{h:,.0f}", ha="center", va="bottom", fontsize=8.5)
ax3.set_xticks(x)
ax3.set_xticklabels([f"{y}년" + ("*" if RES_A["est_flag"].get(y) else "") for y in div_years])
ax3.set_title("③ 연도별 세후 배당금 수령액 추이 (초기 1억 원 기준)", fontsize=12, fontweight="bold")
ax3.set_ylabel("세후 배당금 (만 원)")
ax3.legend(loc="upper left", frameon=True)

# (4) 리밸런싱 구간별 수익률 비교 (매년 4월 -> 다음 해 4월)
ax4 = axes[1, 1]
rebal_dates = sorted(PICKS_A.keys())
periods = [(rebal_dates[i], rebal_dates[i + 1] if i + 1 < len(rebal_dates) else RES_A["nav"].index[-1])
           for i in range(len(rebal_dates))]
p_labels, pr_a, pr_b, pr_k = [], [], [], []
for d0, d1 in periods:
    p_labels.append(f"{d0.strftime('%y.%m')}~{d1.strftime('%y.%m')}")
    pr_a.append((RES_A["nav"].loc[d1] / RES_A["nav"].loc[d0] - 1.0) * 100)
    pr_b.append((RES_B["nav"].loc[d1] / RES_B["nav"].loc[d0] - 1.0) * 100)
    pr_k.append((KOSPI_NAV.loc[d1] / KOSPI_NAV.loc[d0] - 1.0) * 100)

px_idx = np.arange(len(p_labels))
bw = 0.26
ax4.bar(px_idx - bw, pr_a, width=bw, label="전략 A", color=COLOR_A, alpha=0.9)
ax4.bar(px_idx,      pr_b, width=bw, label="전략 B", color=COLOR_B, alpha=0.9)
ax4.bar(px_idx + bw, pr_k, width=bw, label="KOSPI",  color=COLOR_K, alpha=0.8)
ax4.axhline(0, color="black", lw=0.7)
ax4.set_xticks(px_idx)
ax4.set_xticklabels(p_labels, rotation=15, fontsize=9)
ax4.set_title("④ 리밸런싱 구간별 수익률 비교 (4월 → 익년 4월)", fontsize=12, fontweight="bold")
ax4.set_ylabel("구간 수익률 (%)")
ax4.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax4.legend(loc="upper left", frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ==============================================================================
# 9. 연도별 선정 종목 리스트 및 전략 B 배당 함정 필터링 내역
# ==============================================================================
def build_picks_table(picks_dict: dict) -> pd.DataFrame:
    rows = []
    for dt in sorted(picks_dict.keys()):
        for rank_i, c in enumerate(picks_dict[dt], 1):
            rows.append({
                "리밸런싱일": dt.strftime("%Y-%m-%d"),
                "기준FY": f"FY{c['fy']}",
                "순위": rank_i,
                "종목명": c["name"],
                "티커": c["ticker"],
                "경로": c["path"],
                "배당수익률": f"{(c['rebal_yld'] or 0):.2%}",
                "DPS 5년성장": f"{(c['dps_g5'] or 0):.1%}",
                "밸류할인(원형)": f"{(c['val_raw'] or 0):+.1%}",
                "밸류할인(캡)": f"{(c['val_cap'] or 0):+.1%}",
                "52주 낙폭": f"{(c['dd_52w'] or 0):.1%}",
                "60일 거래대금(억)": round((c["turnover_60d"] or 0) / 1e8, 1),
            })
    return pd.DataFrame(rows)

df_picks_a = build_picks_table(PICKS_A)
df_picks_b = build_picks_table(PICKS_B)

print("■ [전략 B — 배당 함정 보완안] 연도별 최종 선정 종목 리스트")
display(df_picks_b)

print("\n■ [전략 A — 현행 규칙] 연도별 선정 종목 요약 (종목명)")
summary_by_yr = pd.DataFrame({
    "전략 A (현행)": {dt.strftime("%Y-%m-%d"): ", ".join(x["name"] for x in PICKS_A[dt]) for dt in sorted(PICKS_A)},
    "전략 B (보완)": {dt.strftime("%Y-%m-%d"): ", ".join(x["name"] for x in PICKS_B[dt]) for dt in sorted(PICKS_B)},
})
display(summary_by_yr)

if FILTERED_OUT_B:
    print(f"\n■ 전략 A 상위권 중 전략 B 함정 필터로 제외된 종목 ({len(FILTERED_OUT_B)}건)")
    display(pd.DataFrame(FILTERED_OUT_B))
else:
    print("\n■ 전략 A 상위권 중 전략 B 함정 필터로 제외된 종목 없음")